# ⚡ Kaggle FN GPU Worker (Multi-LoRA `router` + `generator` + `critic` + NLI)
**Tự động Clone GitHub + Tải Trọng Số từ Google Drive (`gdown`) + Phát Cloudflare Tunnel URL**

Dán URL `https://xxxx.trycloudflare.com` xuất ra ở Bước 3 vào ô **Kaggle Tunnel URL** trên giao diện Web `http://localhost:8000/demo2/`.


In [ ]:
# BƯỚC 1: CLONE GITHUB & TẢI TRỌNG SỐ LORA TỪ GOOGLE DRIVE (KHÔNG CẦN UPLOAD THỦ CÔNG)
import os
import subprocess

!pip install -q fastapi uvicorn gdown transformers peft bitsandbytes accelerate sentence-transformers torch

REPO_URL = "https://github.com/ducmanh-jr/Multi-Agent_RAG.git"
PROJECT_DIR = "/content/Multi-Agent_RAG" if os.path.exists("/content") else "/kaggle/working/Multi-Agent_RAG"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=False)

os.chdir(PROJECT_DIR)

# (Tùy chọn) Điền Google Drive File ID của weights_bundle_v2.zip nếu muốn kéo bản mới nhất từ Drive:
DRIVE_WEIGHTS_FILE_ID = ""
WEIGHTS_DIR = os.path.join(PROJECT_DIR, "training_lab", "weights_drive_bundle")
os.makedirs(WEIGHTS_DIR, exist_ok=True)

if DRIVE_WEIGHTS_FILE_ID:
    import gdown
    zip_path = os.path.join(WEIGHTS_DIR, "weights_bundle.zip")
    gdown.download(id=DRIVE_WEIGHTS_FILE_ID, output=zip_path, quiet=False)
    !unzip -q -o {zip_path} -d {WEIGHTS_DIR}
elif os.path.exists(os.path.join(WEIGHTS_DIR, "weights_bundle.zip")):
    !unzip -q -o {WEIGHTS_DIR}/weights_bundle.zip -d {WEIGHTS_DIR}

os.environ["WEIGHTS_DIR"] = WEIGHTS_DIR
print("[OK] Đã chuẩn bị xong trọng số Multi-LoRA tại:", WEIGHTS_DIR)


In [ ]:
# BƯỚC 2: KHỞI CHẠY KAGGLE MULTI-LORA SERVER & XUẤT LINK CLOUDFLARE TUNNEL
import time
import re
import subprocess

if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "kaggle.kaggle_server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=PROJECT_DIR,
)
time.sleep(8)

tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for _ in range(60):
    line = tunnel_proc.stdout.readline()
    m = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if m:
        print("=" * 75)
        print(f"✅ KAGGLE FN MULTI-LORA WORKER ONLINE: {m.group(0)}")
        print("👉 Hãy copy URL trên dán vào mục 'Kaggle GPU' trong Cài đặt trên Web UI!")
        print("=" * 75)
        break

server_proc.wait()
